## 1. Load and Inspect Silver Taxi Data

Load cleaned NYC Taxi trips from the Silver layer and inspect the schema before building hourly taxi aggregates.

In [ ]:
df_taxi = spark.table("silver_taxi_trips_v2")

print("Taxi Silver schema:")
df_taxi.printSchema()

## 2. Build Hourly Taxi Fact

Filter NYC Taxi trips to the four zones mapped to EPA monitoring stations.

Aggregate trips by pickup date, local hour, and pickup zone to support hourly mobility and air quality analysis.

In [ ]:
from pyspark.sql import functions as F

df_taxi_hourly = (
    df_taxi
    .filter(F.col("PULocationID").isin(31, 147, 135, 192))
    .filter(F.col("tpep_pickup_datetime").isNotNull())
    .withColumn(
        "pickup_date_ny",
        F.to_date("tpep_pickup_datetime")
    )
    .withColumn(
        "hour_ny",
        F.hour("tpep_pickup_datetime")
    )
    .filter(
        F.col("pickup_date_ny").between(
            "2025-06-01", "2025-07-31"
        )
    )
    .groupBy("pickup_date_ny", "hour_ny", "PULocationID")
    .agg(
        F.count("*").alias("hourly_trips")
    )
)

In [ ]:
df_taxi_hourly.show(10, truncate=False)

## 3. Save and Verify Gold Taxi Hourly Fact

Save hourly taxi trip aggregates as a Gold Delta table and verify the persisted records.

In [ ]:
(
    df_taxi_hourly.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_fact_taxi_hourly")
)

df_saved = spark.table("gold_fact_taxi_hourly")

print("Saved rows:", df_saved.count())

df_saved.show(5, truncate=False)

In [ ]:
spark.table("gold_fact_taxi_daily").printSchema()
